In [1]:
"""
Automatically reloads libraries and utilities every time a cell is run
"""
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils.PSF_helpers import *
from utils.Plot_helpers import *
from utils.Zernike_helpers import *
from utils.Booth_helpers import *
from utils.Phase_diversity_helpers_cupy import *

Failed to import cupy.


In [3]:
from scipy.fft import fft2, ifft2
from scipy.fft import fftshift
from scipy.fft import ifftshift

In [4]:
#minor style things to make the figures look nice
plt.style.use("bmh")
plt.rcParams["figure.figsize"] = [548 / 72, 390 / 72]
plt.rcParams["font.size"] = 12
plt.rcParams['text.usetex'] = True
plt.rcParams["figure.dpi"] = 300
import os
os.environ["PATH"] += os.pathsep + "/Library/TeX/texbin"

In [5]:
N_order = 3              # 3 for 3P, 2 for 2P
lambd = 1.3e-3            # Wavelength [mm]
n = 1.333                  # Refractive index
k = 2*n*np.pi/lambd        # Wavenumber
num_apt = 1.05            # Numerical aperture
focal = 7.2                    # Focal length of objective (Olympus) [mm]
mag = 4                   # Magnification rate from input to objective
w_0 = 1000000                 # mm
alpha_three_p = np.arcsin(num_apt / n)

#50 microns
L_ffp = 0.1065
#512 x 512
grid_ffp = 512
grid = Centered_Square_Grid(L_ffp, grid_ffp, 0)
#CONJUGATE BFP LENGTH
grid_bfp = grid_ffp
L_bfp = (lambd * focal * grid_bfp) / L_ffp
three_p_microscope = Microscope(N_order, lambd, n, num_apt, focal, mag, w_0, L_bfp, grid_bfp)
x, y = grid.get_xy()
x_bfp = x * (L_bfp/ L_ffp)
y_bfp = y * (L_bfp/ L_ffp)

In [47]:
N_order = 1              # 3 for 3P, 2 for 2P
lambd = 5.32e-4             # Wavelength [mm]
n = 1.333                  # Refractive index
k = 2*n*np.pi/lambd        # Wavenumber
num_apt = 1.2            # Numerical aperture
focal = 7.2                    # Focal length of objective (Olympus) [mm]
mag = 4                   # Magnification rate from input to objective
w_0 = 1000000                 # mm
alpha_one_p = np.arcsin(num_apt / n)

#50 microns
L_ffp = 0.1065
#512 x 512
grid_ffp = 512
grid = Centered_Square_Grid(L_ffp, grid_ffp, 0)
#CONJUGATE BFP LENGTH
grid_bfp = grid_ffp
L_bfp = (lambd * focal * grid_bfp) / L_ffp
one_p_microscope = Microscope(N_order, lambd, n, num_apt, focal, mag, w_0, L_bfp, grid_bfp)
x, y = grid.get_xy()
x_bfp = x * (L_bfp/ L_ffp)
y_bfp = y * (L_bfp/ L_ffp)

In [48]:
#minor style things to make the figures look nice
plt.style.use("bmh")
plt.rcParams["figure.figsize"] = [548 / 72, 390 / 72]
plt.rcParams["font.size"] = 12
plt.rcParams['text.usetex'] = True
plt.rcParams["figure.dpi"] = 300
import os
os.environ["PATH"] += os.pathsep + "/Library/TeX/texbin"

In [49]:
f = np.load("images/usaf_resolution.npy")

In [51]:
def new_forward_PSF(microscope, grid, aberration):
    x, y, psf = microscope.compute_PSF(grid, aberration, mode = "scalar")

    #normalize the PSF
    return psf

def new_circular_convolve(f: np.array, 
                      psf: np.array):

    S = fft2(ifftshift(psf))
    F = fft2(f, workers = -1)
    convolved_img = np.real(ifft2(S * F, workers = -1))
    return convolved_img

def new_forward_image(microscope: Microscope, 
                  grid: Arbitrary_Grid,
                  f: np.array, 
                  aberration: Aberration):
    
    psf = new_forward_PSF(microscope = microscope,
                      grid = grid,
                       aberration = aberration)


    final_img = new_circular_convolve(f = f, psf = psf)
    return final_img

In [52]:
microscope = one_p_microscope

bias_modes = [[-2,2]]
bias_strength = 1.0
a_stack = [EmptyAberration()] + [Aberration([m], [bias_strength]) for m in bias_modes]

modes_corrected = get_johnson_modes()

rng = np.random.default_rng(15)
true_aberration = generate_johnson_aberration(0.05, alpha_one_p, rng)

d_stack = np.array([new_forward_image(microscope, grid, f, true_aberration + a) for a in a_stack])
